# Freeze NWPU-RESISC45 into GCS (domain swap, step 1)

Same procedure as the Omniglot freeze: download once, verify the publisher's
checksum, repack into the project's convention, compute OUR own SHA256, upload
archive + manifest. Nothing downstream ever downloads from the internet again —
every run reads the frozen archive and records its hash, so a result can always
be traced back to the exact bytes it was trained on.

**Layout produced** (matches what `train_pipeline_entry.py` and
`fsl.validate.validate_remote` already expect):

```
gs://<bucket>/raw/resisc45/resisc45.tar.gz   <- the frozen data
gs://<bucket>/raw/resisc45/MANIFEST.json     <- archive_sha256 + class split + provenance
```

**Why the class split lives in the manifest.** Few-shot needs DISJOINT classes
across train/val/test. That partition is part of the dataset definition, not a
runtime choice — if it drifted between runs, two runs would not be comparable.
So it is decided once, here, and frozen alongside the bytes.

**Dataset:** 31,500 images, 45 scene classes, 700 per class, 256x256 RGB jpg,
ground resolution 0.2-30 m/px. Split 25 train / 10 val / 10 test classes.

**Verified offline:** notebook syntax, the split logic (determinism,
disjointness, sizes) against a synthetic 45-class list, manifest shape.
**NOT verified here:** the download itself and the zip's internal directory
layout — the sandbox has no access to the host. The inventory cell prints what
it actually found; if the top-level folder name differs, fix `EXTRACT_ROOT`
and re-run from there.

## Config + validation

In [1]:
import json
from fsl.validate import validate_config, ConfigError

with open("configs/pipeline_config.json") as f:
    CFG = json.load(f)

try:
    validate_config(CFG)
except ConfigError as e:
    print(e)
    raise

PROJECT = CFG["project"]
BUCKET  = CFG["bucket"]

DATASET   = "resisc45"
SPLIT_SEED = 0          # frozen: changing this invalidates comparability
N_TRAIN, N_VAL, N_TEST = 25, 10, 10

Config OK — project=dark-data-discovery, dataset=omniglot, 5-way 5-shot, HPO off


## Download + verify the publisher's checksum

Pinned to a specific revision (the hash in the URL), so this cell fetches the
same bytes today and in a year. `SOURCE_MD5` is the publisher's checksum — if
it mismatches, stop: the download is corrupt or the source changed.

In [2]:
import hashlib
import urllib.request
from pathlib import Path

SOURCE_URL = ("https://hf.co/datasets/torchgeo/resisc45/resolve/"
              "a826b44d938a883185f11ebe3d512d38b464312f/NWPU-RESISC45.zip")
SOURCE_MD5 = "75206b2e16446591afa88e2628744886"

work = Path("~/data_cache/resisc45").expanduser()
work.mkdir(parents=True, exist_ok=True)
zip_path = work / "NWPU-RESISC45.zip"

if not zip_path.exists():
    print("downloading (a few hundred MB, one time)...")
    urllib.request.urlretrieve(SOURCE_URL, zip_path)
print("downloaded:", zip_path, f"{zip_path.stat().st_size / 1e6:.0f} MB")

h = hashlib.md5()
with open(zip_path, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
actual = h.hexdigest()
assert actual == SOURCE_MD5, (
    f"MD5 mismatch — download is corrupt or the source changed.\n"
    f"  expected {SOURCE_MD5}\n  actual   {actual}")
print("publisher MD5 verified:", actual)

downloading (a few hundred MB, one time)...
downloaded: /home/jupyter/data_cache/resisc45/NWPU-RESISC45.zip 427 MB
publisher MD5 verified: 75206b2e16446591afa88e2628744886


## Extract + inventory

Prints what was actually found. A class here is a directory of images; the
counts should read 45 classes x 700 images. If the top-level folder is named
differently, set `EXTRACT_ROOT` to the printed path and re-run the next cells.

In [3]:
import zipfile

extract_dir = work / "extracted"
if not extract_dir.exists():
    print("extracting...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)

tops = sorted(p for p in extract_dir.iterdir() if p.is_dir())
print("top-level entries:", [p.name for p in tops])
EXTRACT_ROOT = tops[0] if len(tops) == 1 else extract_dir

class_dirs = sorted(p for p in EXTRACT_ROOT.iterdir() if p.is_dir())
counts = {p.name: sum(1 for f in p.iterdir() if f.is_file()) for p in class_dirs}
print(f"\nroot: {EXTRACT_ROOT}")
print(f"classes: {len(class_dirs)}")
print(f"images: {sum(counts.values())}")
print(f"per-class min/max: {min(counts.values())}/{max(counts.values())}")
print("first 5 classes:", list(counts)[:5])

extracting...
top-level entries: ['NWPU-RESISC45']

root: /home/jupyter/data_cache/resisc45/extracted/NWPU-RESISC45
classes: 45
images: 31500
per-class min/max: 700/700
first 5 classes: ['airplane', 'airport', 'baseball_diamond', 'basketball_court', 'beach']


## Freeze the class split

Deterministic from `SPLIT_SEED`: same seed, same partition, forever. Written
into the manifest so any run can be checked against it.

**Note on comparability with published numbers:** papers on few-shot RESISC45
use a 25/10/10 split too, but their exact class lists come from the specific
paper being followed. This split is reproducible and auditable, not identical
to any particular paper — so compare against your own baselines, not against
published accuracies, unless you swap in the reference class list here.

In [4]:
import random

all_classes = sorted(counts)
assert len(all_classes) == N_TRAIN + N_VAL + N_TEST, (
    f"expected {N_TRAIN + N_VAL + N_TEST} classes, found {len(all_classes)}")

shuffled = list(all_classes)
random.Random(SPLIT_SEED).shuffle(shuffled)
split = {
    "train": sorted(shuffled[:N_TRAIN]),
    "val":   sorted(shuffled[N_TRAIN:N_TRAIN + N_VAL]),
    "test":  sorted(shuffled[N_TRAIN + N_VAL:]),
}

# disjointness is the whole point of a few-shot split — assert it
seen = set()
for name, classes in split.items():
    assert not (seen & set(classes)), f"class overlap in {name}"
    seen |= set(classes)
assert seen == set(all_classes)

for name, classes in split.items():
    print(f"{name:5s} ({len(classes):2d}): {', '.join(classes)}")

train (25): airplane, airport, bridge, chaparral, church, cloud, commercial_area, dense_residential, desert, freeway, golf_course, harbor, island, lake, medium_residential, palace, parking_lot, railway, runway, sea_ice, ship, stadium, tennis_court, terrace, thermal_power_station
val   (10): basketball_court, beach, circular_farmland, forest, industrial_area, roundabout, snowberg, sparse_residential, storage_tank, wetland
test  (10): baseball_diamond, ground_track_field, intersection, meadow, mobile_home_park, mountain, overpass, railway_station, rectangular_farmland, river


## Repack into the project's convention + hash it

In [5]:
import tarfile
import time

archive = work / f"{DATASET}.tar.gz"
if not archive.exists():
    print("repacking to tar.gz (a few minutes)...")
    with tarfile.open(archive, "w:gz") as tar:
        for cdir in class_dirs:
            tar.add(cdir, arcname=cdir.name)
print("archive:", archive, f"{archive.stat().st_size / 1e6:.0f} MB")

h = hashlib.sha256()
with open(archive, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
ARCHIVE_SHA = h.hexdigest()
print("archive_sha256:", ARCHIVE_SHA)

repacking to tar.gz (a few minutes)...
archive: /home/jupyter/data_cache/resisc45/resisc45.tar.gz 409 MB
archive_sha256: 764e96932c6f4ed103dfad4dcf382a18f4d37a1bae25b423b08be2a4b5c6255e


## Upload archive + manifest

The manifest is the provenance record: where the bytes came from, the
publisher's checksum, our own hash, the frozen class split, and the image
geometry that the loader and the encoder both need (`in_channels`,
`image_size` — Omniglot is 1x28, this is 3x256 resized to 84).

In [6]:
from google.cloud import storage

manifest = {
    "dataset": DATASET,
    "source_url": SOURCE_URL,
    "source_md5": SOURCE_MD5,
    "archive_sha256": ARCHIVE_SHA,
    "archive_name": f"{DATASET}.tar.gz",
    "frozen_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "n_classes": len(all_classes),
    "n_images": sum(counts.values()),
    "images_per_class": counts,
    "split_seed": SPLIT_SEED,
    "class_split": split,
    "image": {"native_size": 256, "channels": 3, "format": "jpg",
              "train_size": 84},
}

client = storage.Client(project=PROJECT)
bucket = client.bucket(BUCKET)
prefix = f"raw/{DATASET}"

bucket.blob(f"{prefix}/{DATASET}.tar.gz").upload_from_filename(str(archive))
bucket.blob(f"{prefix}/MANIFEST.json").upload_from_string(
    json.dumps(manifest, indent=2), content_type="application/json")
print(f"uploaded to gs://{BUCKET}/{prefix}/")

uploaded to gs://dark-data-discovery-fsl-data/raw/resisc45/


## Verify what landed

In [7]:
back = json.loads(
    bucket.blob(f"{prefix}/MANIFEST.json").download_as_bytes().decode("utf-8"))
assert back["archive_sha256"] == ARCHIVE_SHA
blob = bucket.blob(f"{prefix}/{DATASET}.tar.gz")
blob.reload()
print("manifest round-trips OK")
print(f"archive in GCS: {blob.size / 1e6:.0f} MB")
print(f"classes: {back['n_classes']}, images: {back['n_images']}")
print("split sizes:", {k: len(v) for k, v in back["class_split"].items()})

manifest round-trips OK
archive in GCS: 409 MB
classes: 45, images: 31500
split sizes: {'train': 25, 'val': 10, 'test': 10}


## Next

The data is frozen. Step 2 is the loader (`fsl/data/resisc45.py`) plus the two
hard-coded spots in `loop.py` that currently assume Omniglot: the direct
`from fsl.data.omniglot import ...` and `Conv4(in_c=1, ...)`.